# Trích xuất text game từ file IPA lớn — MeoRocky trên Google Colab

Notebook này chạy **trích xuất text** (bước "Trích xuất" của tool BearTea) bằng
đúng code Python của repo [MunKai0408/MeoRocky](https://github.com/MunKai0408/MeoRocky),
nhưng chạy trên máy ảo Colab thay vì trong tab trình duyệt trên điện thoại.

**Dùng khi nào:** file IPA quá lớn cho Safari trên iPhone (tab chỉ dùng được
~1–1,5 GB RAM nên hay bị văng khi trích file >140 MB). Colab miễn phí có
~11 GB RAM dùng được nên trích được file IPA 2 GB trở lên.

**Cách chạy:** chỉ cần bấm **Run all** — notebook tự tìm file IPA trên Drive
của bạn rồi trích xuất. Kết quả (các file `.json` locale: `en.json`,
`zh-CN.json`, `zh-TW.json`…) được lưu về Google Drive của bạn.

**Game mặc định:** Don't Starve Together 2.1.0 VH (file IPA ~1,69 GB đã có
trên Drive của bạn). Muốn trích game khác thì sửa 2 dòng `GAME_FILE_NAME` và
`DRIVE_FILE_ID` ở ô "Bước 1".

**Lưu ý:**
- Trích xuất đọc theo từng entry trong IPA (streaming), KHÔNG đọc cả file
  IPA vào RAM — RAM đỉnh phụ thuộc entry lớn nhất, không phải tổng dung
  lượng file. File 2 GB+ vẫn chạy được trên Colab miễn phí.
- Notebook này CHỈ trích xuất, chưa dịch. Lấy file `.json` về rồi dịch bằng
  tab Dịch trên web như bình thường.
- File IPA phải nằm trên Google Drive của bạn trước (up lên Drive rồi làm
  theo bước 1 dưới đây).


In [ ]:
# Gắn Google Drive vào máy ảo Colab (sẽ hiện bảng xin quyền truy cập Drive).
from google.colab import drive
drive.mount('/content/drive')
print('Đã gắn Drive tại /content/drive')

In [ ]:
# Clone repo MeoRocky (code trích xuất) về máy ảo + kiểm tra Python.
import os
import subprocess
import sys

REPO_DIR = '/content/MeoRocky'
if not os.path.isfile(os.path.join(REPO_DIR, 'pb3', 'bridge.py')):
    print('Đang clone repo MeoRocky...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/MunKai0408/MeoRocky.git', REPO_DIR],
                   check=True)
else:
    print('Repo đã có sẵn:', REPO_DIR)
print('Python:', sys.version.split()[0])
print('bridge.py:', os.path.isfile(os.path.join(REPO_DIR, 'pb3', 'bridge.py')))

## Bước 1 — Tự động tìm file IPA (bạn không cần gõ gì)

Notebook mặc định trích game **Don't Starve Together 2.1.0 VH** — file IPA
đã có trên Drive của bạn. Ô dưới tự làm 2 việc, theo thứ tự:

1. **Tìm theo tên** trong Drive đã gắn (`MyDrive` + Shared drives).
2. Nếu không thấy (vd file nằm ở "Shared with me", không có trong bản mount),
   **tự tải theo file ID** qua Drive API bằng tài khoản Google của bạn.

Muốn đổi game khác: sửa 2 dòng `GAME_FILE_NAME` và `DRIVE_FILE_ID` ở đầu ô
dưới. Chỉ khi cả 2 cách tự động đều thất bại mới cần điền tay vào
`MANUAL_IPA_PATH` (xem mục Lưu ý).

In [ ]:
# Bước 1 — TỰ ĐỘNG tìm/tải file IPA. Bạn không cần gõ gì ở ô này.
# Game mặc định: Don't Starve Together 2.1.0 VH (file trên Drive của bạn).
# Muốn đổi game khác: sửa GAME_FILE_NAME + DRIVE_FILE_ID dưới đây.
import os
import subprocess

GAME_FILE_NAME = "Don't Starve Together_2.1.0 VH.ipa"
DRIVE_FILE_ID = "1DfcaEn0hEfs88qv2pkO4ed6n4546TsP1"  # ID file IPA này trên Drive
SEARCH_ROOTS = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
DOWNLOAD_PATH = "/content/" + GAME_FILE_NAME
MANUAL_IPA_PATH = ""  # dự phòng cuối: dán đường dẫn mount vào đây nếu 2 cách tự động đều thất bại


def find_ipa_by_name(name, roots):
    """Tìm file theo tên trong Drive đã gắn. Trả đường dẫn đầu tiên tìm
    thấy, hoặc None. Dùng `find ... -print -quit` để dừng ngay khi thấy."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        print("Đang tìm trong:", root)
        r = subprocess.run(["find", root, "-name", name, "-type", "f",
                            "-print", "-quit"],
                           capture_output=True, text=True)
        hits = [h for h in r.stdout.splitlines() if h.strip()]
        if hits:
            return hits[0]
    return None


def download_by_file_id(file_id, dest, service=None):
    """Tải file từ Drive theo file ID (Drive API, tài khoản Google của bạn
    trong Colab). Trả đường dẫn file đã tải, kiểm tra đủ dung lượng.
    `service` chỉ dùng để test (mặc định tự tạo từ tài khoản Colab)."""
    if service is None:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from google.auth import default
        creds, _ = default()
        service = build("drive", "v3", credentials=creds)
    from googleapiclient.http import MediaIoBaseDownload
    meta = service.files().get(fileId=file_id,
                               fields="name,size").execute()
    expect = int(meta.get("size") or 0)
    print("Đang tải: %s (%.1f MB)" % (meta.get("name") or file_id,
                                      expect / 1e6 if expect else 0))
    request = service.files().get_media(fileId=file_id)
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print("  đã tải %.0f%%" % (status.progress() * 100), flush=True)
    got = os.path.getsize(dest)
    if expect and got != expect:
        raise IOError("Tải thiếu: được %d byte, cần %d byte" % (got, expect))
    print("Đã tải xong: %s (%.1f MB)" % (dest, got / 1e6))
    return dest


if MANUAL_IPA_PATH and os.path.isfile(MANUAL_IPA_PATH):
    IPA_PATH = MANUAL_IPA_PATH
    print("Dùng đường dẫn điền tay:", IPA_PATH)
else:
    IPA_PATH = find_ipa_by_name(GAME_FILE_NAME, SEARCH_ROOTS)
    if IPA_PATH:
        print("Đã tìm thấy file tại:", IPA_PATH)
    else:
        print("Không tìm thấy bằng tên trong Drive đã gắn — "
              "chuyển sang tải theo file ID.")
        try:
            IPA_PATH = download_by_file_id(DRIVE_FILE_ID, DOWNLOAD_PATH)
        except Exception as e:  # noqa: BLE001 - rơi về hướng dẫn điền tay
            print("Tải theo file ID thất bại:", e)
            IPA_PATH = None

if not IPA_PATH or not os.path.isfile(IPA_PATH):
    raise FileNotFoundError(
        "Không tự tìm/tải được file IPA.\n"
        "Cách dự phòng: mở khung Files bên trái → drive → MyDrive, tìm file .ipa, "
        "chuột phải → Copy path, dán vào MANUAL_IPA_PATH ở đầu ô này rồi chạy "
        "lại ô này.")
GAME_NAME = os.path.splitext(os.path.basename(IPA_PATH))[0]
OUT_DIR = "/content/drive/MyDrive/MeoRocky-output/%s" % GAME_NAME
print("IPA: %s (%.1f MB)" % (IPA_PATH, os.path.getsize(IPA_PATH) / 1e6))
print("Kết quả sẽ lưu về:", OUT_DIR)

## ⚠️ Lưu ý

- **Phiên Colab tối đa 12 tiếng; không động ~90 phút là tự ngắt; giờ cao điểm
  có thể bị ngắt ngẫu nhiên.** Trích 1 game chỉ mất vài phút–1 tiếng nên thời
  gian dư nhiều, nhưng đừng bấm chạy rồi bỏ đó quá lâu.
- **Kết quả PHẢI lưu về Drive ngay sau khi trích xong** — mọi file trên máy ảo
  Colab mất hết khi phiên tắt, chỉ còn thứ đã lưu về Drive.
- **Đừng tắt tab/đóng trình duyệt khi đang chạy.**
- **RAM dùng được ~11 GB** — notebook xử lý theo từng phần (streaming),
  KHÔNG đọc cả file IPA vào RAM, nên file 2 GB+ vẫn chạy được. Đừng tự ý sửa
  code thành đọc cả file vào RAM.
- **File IPA phải nằm sẵn trên Google Drive trước khi chạy** — đừng up trực
  tiếp file 2 GB từ iPhone lên Colab (lâu, dễ lỗi); up lên Drive trước rồi để
  Colab tải từ Drive.
- **Lần đầu chạy phải bấm cho phép Colab truy cập Drive.**
- **Nếu bị ngắt giữa chừng:** mở lại notebook chạy lại từ đầu là được (trích
  xuất nhanh); kết quả nào đã lưu về Drive thì vẫn còn nguyên.
- **Don't Starve Together 2.1.0 VH (game mặc định) đã thử trên bản mới
  nhất: trích ra 0 file — đây là kết quả ĐÚNG, không phải lỗi notebook.**
  File IPA này KHÔNG phải game Unity (không có `UnityFramework`,
  `data.unity3d`, `.assets` hay `global-metadata.dat` — kiểm tra thật trên
  file 1,69 GB: 8.654 entry, chỉ có dữ liệu engine riêng của Klei như
  `data/movies/*.ogv`, `data/sound/*.fsb`) nên bộ SDK Unity không tìm được
  text nào (275 mục bỏ qua; các file `.po` 17–19 MB bị trần plan 8 MiB
  chặn). Text thật của game nằm trong `data/databundles/scripts.zip`
  (4.067 entry: 4.024 file Lua + 11 file `.po` + `strings.pot`) — và file
  `vietnamese.po` (18.383.078 B) **đã dịch đầy đủ 100%**: 87.862 entry,
  87.829 `msgstr` có nội dung. Nghĩa là game đã có sẵn bản dịch Việt hoàn
  chỉnh trong file — chỉ cần bật tiếng Việt trong game, không cần dịch lại.
  Notebook vẫn đúng và có ích cho các game Unity lớn khác; riêng đường đọc
  `.po` của game Klei chưa có trong tool này.
- **Nếu notebook không tự tìm/tải được file:** mở khung Files bên trái →
  drive → MyDrive, tìm file `.ipa`, chuột phải → Copy path, dán vào
  `MANUAL_IPA_PATH` ở đầu ô "Bước 1" rồi chạy lại ô đó.

In [ ]:
# Bước 2 — Trích xuất (chạy đúng pipeline của tool web, locales en+zh, gộp 1 file/locale).
import hashlib
import json
import resource
import shutil
import sys
import time

t0 = time.monotonic()
sys.path.insert(0, os.path.join(REPO_DIR, 'pb3'))
import bridge

bridge.WEB_ROOT = os.path.join(REPO_DIR, 'pb3')
# Bản web giới hạn file ≤2 GB vì RAM tab trình duyệt; trên Colab không có
# giới hạn đó nên nới trần này (chỉ trong phiên Colab, không sửa repo).
bridge.MAX_WEB_INPUT = 64 * 1024 ** 3

boot = json.loads(bridge.boot())
print('Khởi động:', 'OK' if boot.get('ok') else 'LỖI', '| plugin:', boot.get('plugins'))
assert boot.get('ok'), boot

_last = {'t': 0.0, 'phase': ''}


def _progress(kind, done, total):
    now = time.monotonic()
    if kind != _last['phase'] or now - _last['t'] >= 2.0:
        _last['phase'] = kind
        _last['t'] = now
        if total and total > 0:
            print('  [%s] %d/%d (%.0f%%)' % (kind, done, total, 100.0 * done / total),
                  flush=True)
        else:
            print('  [%s] %d' % (kind, done), flush=True)


bridge.set_progress_cb(_progress)
res = json.loads(bridge.extract(IPA_PATH, locales=['en', 'zh'],
                                merge_one_file=True))
print('Trích xuất:', 'OK' if res.get('ok') else 'LỖI',
      '| quyết định:', res.get('decision'))
if not res.get('ok'):
    print('Lỗi:', res.get('error'))
    raise RuntimeError(res.get('error') or 'trích xuất thất bại')

# Copy file locale thật trong /tmp/pb3out về thư mục trên Drive.
os.makedirs(OUT_DIR, exist_ok=True)
for name in sorted(os.listdir('/tmp/pb3out')):
    src = os.path.join('/tmp/pb3out', name)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(OUT_DIR, name))

print()
print('=== Kết quả ===')
for name in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, name)
    kv = json.load(open(p, encoding='utf-8'))
    h = hashlib.md5(open(p, 'rb').read()).hexdigest()
    print('  %-12s %6d key  md5=%s' % (name, len(kv), h))
for m in res.get('merged_files') or []:
    print('  gộp: locale=%s file=%s keys=%d (trước gộp %d, xung đột %d)'
          % (m.get('locale'), m.get('file'), m.get('keys'),
             m.get('entries_before'), m.get('conflicts_total')))
print('Bỏ qua: %d mục (xem res["skipped"] nếu cần lý do từng mục)'
      % len(res.get('skipped') or []))
dt = time.monotonic() - t0
peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
print('Thời gian: %.1f giây | RAM đỉnh: %.1f MB' % (dt, peak_mb))
print('File đã lưu về Drive tại:', OUT_DIR)

## Xong — nhớ kiểm tra trước khi đóng phiên

- **Kiểm tra file trên Drive:** mở thư mục `MeoRocky-output/<tên game>/` trên
  Google Drive, chắc chắn các file `.json` đã có đủ rồi mới đóng phiên.
- **Phiên Colab tối đa 12 tiếng** — trích 1 game thường chỉ mất vài phút tới
  ~1 tiếng nên dư thời gian. Nhưng **~90 phút không động vào notebook thì
  phiên tự ngắt**; file trên máy ảo Colab sẽ mất, chỉ còn lại những gì đã
  lưu về Drive. Đừng tắt tab khi đang chạy; nếu bị ngắt giữa chừng thì mở lại
  notebook chạy lại từ đầu là được, kết quả đã lưu về Drive vẫn còn nguyên.
- **Nếu trích ra 0 file hoặc ít key lạ** (vd chỉ vài chục key toàn chữ quảng cáo):
  game đó không có bảng ngôn ngữ có cấu trúc (text chính giấu ở chỗ khác) —
  dùng nút "Xem sâu" trên web để dò tiếp, hoặc báo lại để được hỗ trợ.
- **Bước tiếp theo:** lấy file `en.json` vừa trích về, dịch bằng tab Dịch
  trên web như bình thường (notebook này chỉ trích xuất, chưa dịch).

Mở lại notebook này bất cứ lúc nào tại:
https://colab.research.google.com/github/MunKai0408/MeoRocky/blob/main/colab/extract-large-ipa.ipynb
